In [28]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
import scipy.optimize as so

from fractions import Fraction
from IPython.display import display
from scipy.signal import find_peaks
from matplotlib.ticker import AutoMinorLocator

from pyLIMA import event, telescopes
from pyLIMA.models import PSPL_model
from pyLIMA.simulations import simulator


# ============================================================
# ESTILO DE FIGURA PARA PAPER
# ============================================================

def set_paper_style():

    plt.rcParams.update({

        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 11,
        "legend.fontsize": 10,

        "xtick.labelsize": 11,
        "ytick.labelsize": 11,

        "axes.linewidth": 0.8,

        "xtick.direction": "in",
        "ytick.direction": "in",

        "xtick.top": True,
        "ytick.right": True,

        "xtick.major.size": 5,
        "ytick.major.size": 5,

        "xtick.minor.size": 3,
        "ytick.minor.size": 3,

        "xtick.major.width": 0.9,
        "ytick.major.width": 0.9,

        "xtick.minor.width": 0.7,
        "ytick.minor.width": 0.7,

        "savefig.dpi": 300,
        "savefig.bbox": "tight",

        "mathtext.fontset": "stix",
        "font.family": "STIXGeneral",
    })


# ============================================================
# AJUSTE PSPL
# ============================================================

def chi2_theoretical(
    fit_params,
    target_model,
    A_true,
    fs_fixed=1.0,
    ftotal_fixed=1.0,
):

    fit_params = np.asarray(
        fit_params,
        dtype=float,
    )

    full_params = np.concatenate([
        fit_params,
        [
            fs_fixed,
            ftotal_fixed,
        ],
    ])

    py_params = (
        target_model.compute_pyLIMA_parameters(
            full_params
        )
    )

    telescope = (
        target_model.event.telescopes[0]
    )

    model_pred = (
        target_model.model_magnification(
            telescope,
            py_params,
        )
    )

    resid = (
        A_true
        - model_pred
    )

    return float(
        np.sum(
            resid**2
        )
    )


def fit_pspl(
    ev,
    A_truth,
    t0_true,
    u0_true,
    tE_true,
    fs,
    ftotal,
):

    model_pspl = PSPL_model.PSPLmodel(

        ev,

        parallax=[
            "None",
            0.0,
        ],

        double_source=[
            "None",
            0.0,
        ],
    )

    model_pspl.define_model_parameters()

    x0 = np.array(
        [
            t0_true,
            u0_true,
            tE_true,
        ],
        dtype=float,
    )

    res = so.minimize(

        chi2_theoretical,

        x0=x0,

        args=(
            model_pspl,
            A_truth,
            fs,
            ftotal,
        ),

        method="Nelder-Mead",

        options=dict(
            maxiter=200000,
            xatol=1e-10,
            fatol=1e-6,
        ),
    )

    if not res.success:

        raise RuntimeError(
            f"Fit failed: {res.message}"
        )

    best = np.asarray(
        res.x,
        dtype=float,
    )

    best_full = np.concatenate([
        best,
        [
            fs,
            ftotal,
        ],
    ])

    py_best = (
        model_pspl.compute_pyLIMA_parameters(
            best_full
        )
    )

    A_fit = (
        model_pspl.model_magnification(
            ev.telescopes[0],
            py_best,
        )
    )

    # ========================================================
    # Trayectoria PSPL ajustada
    # ========================================================

    traj_fit = (
        model_pspl.sources_trajectory(

            model_pspl.event.telescopes[0],

            py_best,

            data_type="photometry",
        )
    )

    fit_x = np.asarray(
        traj_fit[0],
        dtype=float,
    )

    fit_y = np.asarray(
        traj_fit[1],
        dtype=float,
    )

    return dict(

        A_fit=A_fit,

        best_fit=best,

        model=model_pspl,

        py_params=py_best,

        x=fit_x,

        y=fit_y,
    )


# ============================================================
# FUNCIONES BÁSICAS
# ============================================================

def mag_to_flux(
    mag_value,
    zp=0.0,
):

    return 10**(
        -0.4
        * (
            mag_value
            - zp
        )
    )


def mag(
    zp,
    Flux,
):

    return (
        zp
        - 2.5
        * np.log10(
            np.abs(
                Flux
            )
        )
    )


def build_sim_event(
    t,
    mag0=19.0,
    emag=1e-9,
    filt="G",
):

    ev = event.Event()

    ev.name = "Simulated"

    ev.ra = 170
    ev.dec = -70

    lc = np.c_[

        t,

        np.full_like(
            t,
            mag0,
        ),

        np.full_like(
            t,
            emag,
        ),
    ]

    tel = telescopes.Telescope(

        name="Simulation",

        camera_filter=filt,

        lightcurve=lc.astype(
            float
        ),

        lightcurve_names=[
            "time",
            "mag",
            "err_mag",
        ],

        lightcurve_units=[
            "JD",
            "mag",
            "mag",
        ],

        location="Earth",
    )

    ev.telescopes.append(
        tel
    )

    return ev


# ============================================================
# FÍSICA
# ============================================================

def thetaE_from_lens_mass(
    M_lens_Msun,
    Dl_kpc=4.0,
    Ds_kpc=8.0,
):

    if (
        Dl_kpc <= 0
        or
        Ds_kpc <= 0
    ):

        raise ValueError(
            "Dl_kpc y Ds_kpc deben ser positivos."
        )

    if Dl_kpc >= Ds_kpc:

        raise ValueError(
            "Debe cumplirse Dl_kpc < Ds_kpc."
        )

    kappa_mas_per_Msun = 8.144

    pi_rel_mas = (
        1.0 / Dl_kpc
        - 1.0 / Ds_kpc
    )

    thetaE_mas = np.sqrt(
        kappa_mas_per_Msun
        * M_lens_Msun
        * pi_rel_mas
    )

    return (
        thetaE_mas,
        pi_rel_mas,
    )


def kepler_binary_source_xiE(
    P_days,
    q_mass,
    Mtot_source_Msun=1.8,
    M_lens_Msun=0.3,
    Dl_kpc=4.0,
    Ds_kpc=8.0,
    tE_days=20.0,
    xi_source="source1",
):

    P_yr = (
        P_days
        / 365.25
    )

    # ========================================================
    # Masas
    # ========================================================

    M1_Msun = (
        Mtot_source_Msun
        / (
            1.0
            + q_mass
        )
    )

    M2_Msun = (
        q_mass
        * M1_Msun
    )

    # ========================================================
    # Semiejes
    # ========================================================

    a_rel_AU = (
        Mtot_source_Msun
        * P_yr**2
    )**(
        1.0 / 3.0
    )

    a1_AU = (
        a_rel_AU
        * M2_Msun
        / Mtot_source_Msun
    )

    a2_AU = (
        a_rel_AU
        * M1_Msun
        / Mtot_source_Msun
    )

    thetaE_mas, pi_rel_mas = (
        thetaE_from_lens_mass(

            M_lens_Msun=M_lens_Msun,

            Dl_kpc=Dl_kpc,

            Ds_kpc=Ds_kpc,
        )
    )

    mu_rel_implied_masyr = (
        365.25
        * thetaE_mas
        / tE_days
    )

    RE_source_AU = (
        thetaE_mas
        * Ds_kpc
    )

    if xi_source == "source1":

        a_source_AU = (
            a1_AU
        )

    elif xi_source == "source2":

        a_source_AU = (
            a2_AU
        )

    elif xi_source == "relative":

        a_source_AU = (
            a_rel_AU
        )

    else:

        raise ValueError(
            "xi_source debe ser "
            "'source1', 'source2' o 'relative'."
        )

    xiE = (
        a_source_AU
        / RE_source_AU
    )

    return xiE, dict(

        P_days=P_days,
        P_yr=P_yr,

        Mtot_source_Msun=Mtot_source_Msun,

        M1_Msun=M1_Msun,
        M2_Msun=M2_Msun,

        M_lens_Msun=M_lens_Msun,

        q_mass=q_mass,

        a_rel_AU=a_rel_AU,

        a1_AU=a1_AU,

        a2_AU=a2_AU,

        thetaE_mas=thetaE_mas,

        pi_rel_mas=pi_rel_mas,

        Dl_kpc=Dl_kpc,

        Ds_kpc=Ds_kpc,

        RE_source_AU=RE_source_AU,

        tE_model_days=tE_days,

        mu_rel_implied_masyr=(
            mu_rel_implied_masyr
        ),

        xi_source=xi_source,

        xiE=xiE,
    )


# ============================================================
# MODELO FUENTE BINARIA
# ============================================================

def build_binary_source_state(
    t,
    P_days,
    t0_true,
    u0_true,
    tE_true,
    xiE,
    use_kepler,
    Mtot_source_Msun,
    M_lens_Msun,
    Dl_kpc,
    Ds_kpc,
    xi_source,
    theta,
    phi0,
    lambda_xi,
    q_mass,
    qflux,
    ms=24.0,
    mtotal=24.0,
):

    omega = (
        2.0
        * np.pi
        / float(
            P_days
        )
    )

    # ========================================================
    # xi_E
    # ========================================================

    if use_kepler:

        xiE_use, kepler_info = (
            kepler_binary_source_xiE(

                P_days=P_days,

                q_mass=q_mass,

                Mtot_source_Msun=Mtot_source_Msun,

                M_lens_Msun=M_lens_Msun,

                Dl_kpc=Dl_kpc,

                Ds_kpc=Ds_kpc,

                tE_days=tE_true,

                xi_source=xi_source,
            )
        )

    else:

        xiE_use = xiE

        kepler_info = None

    xi_para = (
        xiE_use
        * np.cos(
            theta
        )
    )

    xi_perp = (
        xiE_use
        * np.sin(
            theta
        )
    )

    # ========================================================
    # Evento
    # ========================================================

    ev = build_sim_event(
        t,
        mag0=19.0,
        emag=1e-9,
        filt="G",
    )

    # ========================================================
    # Modelo Xallarap
    # ========================================================

    model_xal = (
        PSPL_model.PSPLmodel(

            ev,

            parallax=[
                "None",
                0.0,
            ],

            double_source=[
                "Circular",
                t0_true,
            ],
        )
    )

    model_xal.define_model_parameters()

    # ========================================================
    # Flujos
    # ========================================================

    ZP = 27.615

    fs = mag_to_flux(
        ms,
        zp=ZP,
    )

    ftotal = mag_to_flux(
        mtotal,
        zp=ZP,
    )

    params_xal = [

        t0_true,

        u0_true,

        tE_true,

        xi_para,

        xi_perp,

        omega,

        phi0,

        lambda_xi,

        q_mass,

        qflux,

        fs,

        ftotal,
    ]

    py_params_xal = (
        model_xal.compute_pyLIMA_parameters(
            params_xal
        )
    )

    simulator.simulate_lightcurve(

        model_xal,

        py_params_xal,
    )

    # ========================================================
    # Magnificación
    # ========================================================

    A_truth = (

        model_xal.model_magnification(

            ev.telescopes[0],

            py_params_xal,
        )

        / (
            1.0
            + qflux
        )
    )

    F_truth = (

        model_xal.compute_the_microlensing_model(

            ev.telescopes[0],

            py_params_xal,

        )["photometry"]
    )

    ev.telescopes[0].lightcurve[
        "flux"
    ] = A_truth

    ev.telescopes[0].lightcurve[
        "mag"
    ] = mag(
        ZP,
        F_truth,
    )

    # ========================================================
    # Trayectorias
    # ========================================================

    trajectories = (
        model_xal.sources_trajectory(

            model_xal.event.telescopes[0],

            py_params_xal,

            data_type="photometry",
        )
    )

    (
        source1_x,
        source1_y,

        source2_x,
        source2_y,

        dsep,
        dalpha,

    ) = trajectories

    return dict(

        t=np.asarray(
            t
        ),

        event=ev,

        model=model_xal,

        py_params=py_params_xal,

        A=A_truth,

        F=F_truth,

        fs=fs,

        ftotal=ftotal,

        source1_x=source1_x,

        source1_y=source1_y,

        source2_x=source2_x,

        source2_y=source2_y,

        dseparation=dsep,

        dalpha=dalpha,

        omega=omega,

        xiE_use=xiE_use,

        xi_para=xi_para,

        xi_perp=xi_perp,

        kepler_info=kepler_info,
    )


# ============================================================
# t_dev = FWHM DE LA FEATURE DOMINANTE
# ============================================================

def measure_residual_fwhm(
    t,
    residual,
):

    t = np.asarray(
        t,
        dtype=float,
    )

    residual = np.asarray(
        residual,
        dtype=float,
    )

    abs_residual = np.abs(
        residual
    )

    finite = (
        np.isfinite(t)
        &
        np.isfinite(
            abs_residual
        )
    )

    if np.sum(
        finite
    ) < 2:

        return dict(
            Rmax=np.nan,
            half_max=np.nan,
            i_max=None,
            t_max=np.nan,
            t_left=np.nan,
            t_right=np.nan,
            t_dev_days=np.nan,
            truncated_left=False,
            truncated_right=False,
            truncated=False,
        )

    safe_abs = np.where(
        finite,
        abs_residual,
        -np.inf,
    )

    i_max = int(
        np.argmax(
            safe_abs
        )
    )

    Rmax = (
        abs_residual[
            i_max
        ]
    )

    if (
        not np.isfinite(
            Rmax
        )
        or
        Rmax <= 0
    ):

        return dict(
            Rmax=Rmax,
            half_max=0.0,
            i_max=i_max,
            t_max=t[i_max],
            t_left=t[i_max],
            t_right=t[i_max],
            t_dev_days=0.0,
            truncated_left=False,
            truncated_right=False,
            truncated=False,
        )

    half_max = (
        0.5
        * Rmax
    )

    # ========================================================
    # IZQUIERDA
    # ========================================================

    i_left = (
        i_max
    )

    while (
        i_left > 0
        and
        np.isfinite(
            abs_residual[
                i_left - 1
            ]
        )
        and
        abs_residual[
            i_left - 1
        ] >= half_max
    ):

        i_left -= 1

    if i_left == 0:

        t_left = (
            t[0]
        )

        truncated_left = (
            abs_residual[0]
            >= half_max
        )

    else:

        i0 = (
            i_left - 1
        )

        i1 = (
            i_left
        )

        t0 = t[i0]
        t1 = t[i1]

        y0 = (
            abs_residual[i0]
        )

        y1 = (
            abs_residual[i1]
        )

        if np.isclose(
            y1,
            y0,
        ):

            t_left = (
                t1
            )

        else:

            t_left = (
                t0
                +
                (
                    half_max
                    - y0
                )
                *
                (
                    t1
                    - t0
                )
                /
                (
                    y1
                    - y0
                )
            )

        truncated_left = (
            False
        )

    # ========================================================
    # DERECHA
    # ========================================================

    i_right = (
        i_max
    )

    while (
        i_right < len(t) - 1
        and
        np.isfinite(
            abs_residual[
                i_right + 1
            ]
        )
        and
        abs_residual[
            i_right + 1
        ] >= half_max
    ):

        i_right += 1

    if i_right == len(t) - 1:

        t_right = (
            t[-1]
        )

        truncated_right = (
            abs_residual[-1]
            >= half_max
        )

    else:

        i0 = (
            i_right
        )

        i1 = (
            i_right + 1
        )

        t0 = t[i0]
        t1 = t[i1]

        y0 = (
            abs_residual[i0]
        )

        y1 = (
            abs_residual[i1]
        )

        if np.isclose(
            y1,
            y0,
        ):

            t_right = (
                t0
            )

        else:

            t_right = (
                t0
                +
                (
                    half_max
                    - y0
                )
                *
                (
                    t1
                    - t0
                )
                /
                (
                    y1
                    - y0
                )
            )

        truncated_right = (
            False
        )

    t_dev_days = (
        t_right
        - t_left
    )

    truncated = (
        truncated_left
        or
        truncated_right
    )

    return dict(

        Rmax=Rmax,

        half_max=half_max,

        i_max=i_max,

        t_max=t[
            i_max
        ],

        t_left=t_left,

        t_right=t_right,

        t_dev_days=t_dev_days,

        truncated_left=truncated_left,

        truncated_right=truncated_right,

        truncated=truncated,
    )


# ============================================================
# MODELO COMPLETO + AJUSTE
# ============================================================

def compute_binary_source_case(
    P_days=800.0,
    t0_true=0.0,
    u0_true=0.01,
    tE_true=20.0,
    xiE=0.3,
    use_kepler=True,
    Mtot_source_Msun=1.8,
    M_lens_Msun=0.3,
    Dl_kpc=4.0,
    Ds_kpc=8.0,
    xi_source="source1",
    theta=0.0,
    phi0=np.pi,
    lambda_xi=0.0,
    q_mass=0.8,
    qflux=0.45,
    ms=24.0,
    mtotal=24.0,
    window_k=5.0,
    n_points=5000,
):

    t = np.linspace(

        t0_true
        - window_k
        * tE_true,

        t0_true
        + window_k
        * tE_true,

        int(
            n_points
        ),
    )

    state = (
        build_binary_source_state(

            t=t,

            P_days=P_days,

            t0_true=t0_true,

            u0_true=u0_true,

            tE_true=tE_true,

            xiE=xiE,

            use_kepler=use_kepler,

            Mtot_source_Msun=Mtot_source_Msun,

            M_lens_Msun=M_lens_Msun,

            Dl_kpc=Dl_kpc,

            Ds_kpc=Ds_kpc,

            xi_source=xi_source,

            theta=theta,

            phi0=phi0,

            lambda_xi=lambda_xi,

            q_mass=q_mass,

            qflux=qflux,

            ms=ms,

            mtotal=mtotal,
        )
    )

    # ========================================================
    # Ajuste PSPL
    # ========================================================

    fit = fit_pspl(

        state[
            "event"
        ],

        state[
            "A"
        ],

        t0_true,

        u0_true,

        tE_true,

        state[
            "fs"
        ],

        state[
            "ftotal"
        ],
    )

    A_fit = (
        fit[
            "A_fit"
        ]
    )

    best_fit = (
        fit[
            "best_fit"
        ]
    )

    t0_fit = (
        best_fit[0]
    )

    u0_fit = (
        best_fit[1]
    )

    tE_fit = (
        best_fit[2]
    )

    # ========================================================
    # Residuales
    # ========================================================

    resid_A = (
        A_fit
        - state[
            "A"
        ]
    )

    abs_resid_A = (
        np.abs(
            resid_A
        )
    )

    rms_abs_resid_A = np.sqrt(
        np.mean(
            abs_resid_A**2
        )
    )

    max_abs_resid_A = (
        np.max(
            abs_resid_A
        )
    )

    # ========================================================
    # t_dev
    # ========================================================

    dev_info = (
        measure_residual_fwhm(

            t=state[
                "t"
            ],

            residual=resid_A,
        )
    )

    return dict(

        t=state[
            "t"
        ],

        A=state[
            "A"
        ],

        F=state[
            "F"
        ],

        A_fit=A_fit,

        resid_A=resid_A,

        abs_resid_A=abs_resid_A,

        rms_abs_resid_A=(
            rms_abs_resid_A
        ),

        max_abs_resid_A=(
            max_abs_resid_A
        ),

        # ----------------------------------------------------
        # t_dev
        # ----------------------------------------------------

        t_dev_days=(
            dev_info[
                "t_dev_days"
            ]
        ),

        t_dev_left=(
            dev_info[
                "t_left"
            ]
        ),

        t_dev_right=(
            dev_info[
                "t_right"
            ]
        ),

        t_dev_tmax=(
            dev_info[
                "t_max"
            ]
        ),

        t_dev_halfmax=(
            dev_info[
                "half_max"
            ]
        ),

        t_dev_truncated=(
            dev_info[
                "truncated"
            ]
        ),

        # ----------------------------------------------------
        # PSPL fit
        # ----------------------------------------------------

        best_fit=best_fit,

        t0_fit=t0_fit,

        u0_fit=u0_fit,

        tE_fit=tE_fit,

        fit_x=fit[
            "x"
        ],

        fit_y=fit[
            "y"
        ],

        # ----------------------------------------------------
        # Truth
        # ----------------------------------------------------

        t0_true=t0_true,

        u0_true=u0_true,

        tE_true=tE_true,

        delta_t0=(
            t0_fit
            - t0_true
        ),

        delta_u0=(
            u0_fit
            - u0_true
        ),

        delta_tE=(
            tE_fit
            - tE_true
        ),

        # ----------------------------------------------------
        # Modelo Xallarap
        # ----------------------------------------------------

        model=state[
            "model"
        ],

        py_params=state[
            "py_params"
        ],

        source1_x=state[
            "source1_x"
        ],

        source1_y=state[
            "source1_y"
        ],

        source2_x=state[
            "source2_x"
        ],

        source2_y=state[
            "source2_y"
        ],

        dseparation=state[
            "dseparation"
        ],

        dalpha=state[
            "dalpha"
        ],

        omega=state[
            "omega"
        ],

        xiE_use=state[
            "xiE_use"
        ],

        xi_para=state[
            "xi_para"
        ],

        xi_perp=state[
            "xi_perp"
        ],

        kepler_info=state[
            "kepler_info"
        ],

        fs=state[
            "fs"
        ],

        ftotal=state[
            "ftotal"
        ],
    )


# ============================================================
# TRAYECTORIA PSPL EXTENDIDA
# ============================================================

def compute_pspl_trajectory_from_fit(
    t,
    best_fit,
    fs,
    ftotal,
):

    ev = build_sim_event(

        np.asarray(
            t
        ),

        mag0=19.0,

        emag=1e-9,

        filt="G",
    )

    model_pspl = (
        PSPL_model.PSPLmodel(

            ev,

            parallax=[
                "None",
                0.0,
            ],

            double_source=[
                "None",
                0.0,
            ],
        )
    )

    model_pspl.define_model_parameters()

    full_params = np.concatenate([

        np.asarray(
            best_fit,
            dtype=float,
        ),

        [
            fs,
            ftotal,
        ],
    ])

    py_params = (
        model_pspl.compute_pyLIMA_parameters(
            full_params
        )
    )

    trajectories = (
        model_pspl.sources_trajectory(

            model_pspl.event.telescopes[0],

            py_params,

            data_type="photometry",
        )
    )

    x = np.asarray(
        trajectories[0],
        dtype=float,
    )

    y = np.asarray(
        trajectories[1],
        dtype=float,
    )

    return (
        x,
        y,
    )


# ============================================================
# HELPERS GRÁFICOS
# ============================================================

def _arr(
    x
):

    return np.asarray(
        getattr(
            x,
            "value",
            x,
        )
    )


def add_direction_arrows(
    ax,
    x,
    y,
    color,
    n_arrows=5,
    scale=0.08,
    lw=1.2,
    zorder=15,
):

    if (
        x is None
        or
        y is None
    ):

        return

    x = _arr(
        x
    )

    y = _arr(
        y
    )

    if len(
        x
    ) < 3:

        return

    idxs = np.linspace(

        1,

        len(x) - 2,

        n_arrows,

        dtype=int,
    )

    for idx in idxs:

        dx = (
            x[
                idx + 1
            ]
            - x[
                idx - 1
            ]
        )

        dy = (
            y[
                idx + 1
            ]
            - y[
                idx - 1
            ]
        )

        norm = np.hypot(
            dx,
            dy,
        )

        if norm == 0:

            continue

        ax.annotate(

            "",

            xy=(
                x[idx]
                + scale
                * dx
                / norm,

                y[idx]
                + scale
                * dy
                / norm,
            ),

            xytext=(
                x[idx],
                y[idx],
            ),

            arrowprops=dict(

                arrowstyle="->",

                mutation_scale=9,

                color=color,

                lw=lw,
            ),

            zorder=zorder,
        )


# ============================================================
# PARÁMETROS
# ============================================================

def build_param_text(
    out,
    P_days,
    tE_true,
    u0_true,
    xiE,
    use_kepler,
    Mtot_source_Msun,
    M_lens_Msun,
    q_mass,
    qflux,
):

    kepler_info = (
        out[
            "kepler_info"
        ]
    )

    xiE_plot = (
        out[
            "xiE_use"
        ]
    )

    if (
        use_kepler
        and
        kepler_info is not None
    ):

        param_text = (

            rf"$P={P_days:.3g}\,\mathrm{{d}}$"
            "\n"

            rf"$t_E={tE_true:.3g}\,\mathrm{{d}}$"
            "\n"

            rf"$u_0={u0_true:.3g}$"
            "\n"

            rf"$M_{{\rm tot}}={Mtot_source_Msun:.3g}\,M_\odot$"
            "\n"

            rf"$q_M={q_mass:.3g}$"
            "\n"

            rf"$\pi_{{\rm rel}}="
            rf"{kepler_info['pi_rel_mas']:.3g}"
            r"\,\mathrm{mas}$"
            "\n"

            rf"$\theta_E="
            rf"{kepler_info['thetaE_mas']:.3g}"
            r"\,\mathrm{mas}$"
            "\n"

            rf"$R_{{E,S}}="
            rf"{kepler_info['RE_source_AU']:.3g}"
            r"\,\mathrm{AU}$"
            "\n"

            rf"$\xi_E={xiE_plot:.3g}$"
            "\n"

            rf"$q_F={qflux:.3g}$"
        )

    else:

        param_text = (

            rf"$P={P_days:.3g}\,\mathrm{{d}}$"
            "\n"

            rf"$t_E={tE_true:.3g}\,\mathrm{{d}}$"
            "\n"

            rf"$u_0={u0_true:.3g}$"
            "\n"

            rf"$\xi_E={xiE:.3g}$"
            "\n"

            rf"$M_{{\rm tot}}={Mtot_source_Msun:.3g}\,M_\odot$"
            "\n"

            rf"$q_M={q_mass:.3g}$"
            "\n"

            rf"$q_F={qflux:.3g}$"
        )

    return (
        param_text
    )


# ============================================================
# TICKS TEMPORALES DINÁMICOS
# ============================================================

def _time_offset_label(
    x
):

    if np.isclose(
        x,
        0.0,
    ):

        return r"$t_0$"

    sign = (
        "+"
        if x > 0
        else "-"
    )

    a = abs(
        x
    )

    f = Fraction(
        float(
            a
        )
    ).limit_denominator(
        12
    )

    num = (
        f.numerator
    )

    den = (
        f.denominator
    )

    if den == 1:

        if num == 1:

            term = (
                r"t_E"
            )

        else:

            term = (
                rf"{num}t_E"
            )

    else:

        if num == 1:

            term = (
                rf"\frac{{1}}{{{den}}}t_E"
            )

        else:

            term = (
                rf"\frac{{{num}}}{{{den}}}t_E"
            )

    return (
        rf"$t_0{sign}{term}$"
    )


def set_time_ticks(
    ax,
    t0_true,
    tE_true,
    window_k,
    max_labels=7,
):

    nice_steps = np.array([
        0.10,
        0.125,
        0.20,
        0.25,
        1/3,
        0.50,
        1.0,
        2.0,
        2.5,
        5.0,
        10.0,
        20.0,
        50.0,
    ])

    chosen_step = (
        nice_steps[
            -1
        ]
    )

    for step in nice_steps:

        n_ticks = (
            2
            * int(
                np.floor(
                    window_k
                    / step
                )
            )
            + 1
        )

        if n_ticks <= max_labels:

            chosen_step = (
                step
            )

            break

    n_side = int(
        np.floor(
            window_k
            / chosen_step
            + 1e-10
        )
    )

    offsets = (
        np.arange(
            -n_side,
            n_side + 1
        )
        * chosen_step
    )

    ticks = (
        t0_true
        + offsets
        * tE_true
    )

    labels = [

        _time_offset_label(
            x
        )

        for x in offsets
    ]

    ax.set_xticks(
        ticks
    )

    ax.set_xticklabels(
        labels,
        fontsize=11,
    )


# ============================================================
# TIEMPOS DE MÍNIMA SEPARACIÓN AL LENTE
# ============================================================

def measure_binary_source_peak_delay(
    out,
):

    t = _arr(
        out[
            "t"
        ]
    )

    r1 = np.column_stack([

        _arr(
            out[
                "source1_x"
            ]
        ),

        _arr(
            out[
                "source1_y"
            ]
        ),
    ])

    r2 = np.column_stack([

        _arr(
            out[
                "source2_x"
            ]
        ),

        _arr(
            out[
                "source2_y"
            ]
        ),
    ])

    u1 = np.linalg.norm(
        r1,
        axis=1,
    )

    u2 = np.linalg.norm(
        r2,
        axis=1,
    )

    i1 = np.nanargmin(
        u1
    )

    i2 = np.nanargmin(
        u2
    )

    return dict(

        t_peak_geom_1=(
            t[
                i1
            ]
        ),

        t_peak_geom_2=(
            t[
                i2
            ]
        ),
    )


# ============================================================
# PLOT PRINCIPAL
# ============================================================

def plot_binary_source_widget(
    P_days=800.0,
    t0_true=0.0,
    u0_true=0.01,
    tE_true=20.0,
    xiE=0.3,
    use_kepler=True,
    Mtot_source_Msun=1.8,
    M_lens_Msun=0.3,
    Dl_kpc=4.0,
    Ds_kpc=8.0,
    xi_source="source1",
    theta=0.0,
    phi0=np.pi,
    lambda_xi=0.0,
    q_mass=0.8,
    qflux=0.45,
    window_k=1.0,
    n_points=5000,
    traj_lim=1.0,
    logA=False,
):

    plt.close(
        "all"
    )

    set_paper_style()

    # ========================================================
    # Convención visual
    # ========================================================

    COLOR_TRUE = (
        "C0"
    )

    COLOR_FIT = (
        "red"
    )

    LS_SOURCE1 = (
        "-"
    )

    LS_SOURCE2 = (
        "--"
    )

    LS_FIT = (
        "-"
    )

    # ========================================================
    # Line widths
    #
    # Modelo verdadero más grueso.
    # Fit más fino y por delante.
    # ========================================================

    LW_TRUE = 2.0

    LW_FIT = 1.0

    LW_TRUE_EXT = 0.8

    LW_FIT_EXT = 0.5

    # ========================================================
    # Modelo principal
    # ========================================================

    out = (
        compute_binary_source_case(

            P_days=P_days,

            t0_true=t0_true,

            u0_true=u0_true,

            tE_true=tE_true,

            xiE=xiE,

            use_kepler=use_kepler,

            Mtot_source_Msun=Mtot_source_Msun,

            M_lens_Msun=M_lens_Msun,

            Dl_kpc=Dl_kpc,

            Ds_kpc=Ds_kpc,

            xi_source=xi_source,

            theta=theta,

            phi0=phi0,

            lambda_xi=lambda_xi,

            q_mass=q_mass,

            qflux=qflux,

            ms=24.0,

            mtotal=24.0,

            window_k=window_k,

            n_points=n_points,
        )
    )

    # ========================================================
    # Trayectoria extendida
    #
    # t0 +- 10 tE
    # ========================================================

    t_long = np.linspace(

        t0_true
        - 10.0
        * tE_true,

        t0_true
        + 10.0
        * tE_true,

        max(
            8000,
            int(
                n_points
            ),
        ),
    )

    long_state = (
        build_binary_source_state(

            t=t_long,

            P_days=P_days,

            t0_true=t0_true,

            u0_true=u0_true,

            tE_true=tE_true,

            xiE=xiE,

            use_kepler=use_kepler,

            Mtot_source_Msun=Mtot_source_Msun,

            M_lens_Msun=M_lens_Msun,

            Dl_kpc=Dl_kpc,

            Ds_kpc=Ds_kpc,

            xi_source=xi_source,

            theta=theta,

            phi0=phi0,

            lambda_xi=lambda_xi,

            q_mass=q_mass,

            qflux=qflux,

            ms=24.0,

            mtotal=24.0,
        )
    )

    # ========================================================
    # PSPL extendido
    # ========================================================

    fit_x_long, fit_y_long = (
        compute_pspl_trajectory_from_fit(

            t=t_long,

            best_fit=out[
                "best_fit"
            ],

            fs=out[
                "fs"
            ],

            ftotal=out[
                "ftotal"
            ],
        )
    )

    # ========================================================
    # Arrays
    # ========================================================

    t = _arr(
        out[
            "t"
        ]
    )

    A = _arr(
        out[
            "A"
        ]
    )

    A_fit = _arr(
        out[
            "A_fit"
        ]
    )

    resid_A = _arr(
        out[
            "resid_A"
        ]
    )

    s1x = _arr(
        out[
            "source1_x"
        ]
    )

    s1y = _arr(
        out[
            "source1_y"
        ]
    )

    s2x = _arr(
        out[
            "source2_x"
        ]
    )

    s2y = _arr(
        out[
            "source2_y"
        ]
    )

    fit_x = _arr(
        out[
            "fit_x"
        ]
    )

    fit_y = _arr(
        out[
            "fit_y"
        ]
    )

    s1x_long = _arr(
        long_state[
            "source1_x"
        ]
    )

    s1y_long = _arr(
        long_state[
            "source1_y"
        ]
    )

    s2x_long = _arr(
        long_state[
            "source2_x"
        ]
    )

    s2y_long = _arr(
        long_state[
            "source2_y"
        ]
    )

    delay_info = (
        measure_binary_source_peak_delay(
            out
        )
    )

    # ========================================================
    # FIGURA
    # ========================================================

    fig = plt.figure(
        figsize=(
            9.0,
            5.8,
        ),dpi=200
    )

    gs = fig.add_gridspec(

        2,
        1,

        height_ratios=[
            3.3,
            1.0,
        ],

        hspace=0.04,
    )

    axA = fig.add_subplot(
        gs[
            0,
            0
        ]
    )

    axR = fig.add_subplot(

        gs[
            1,
            0
        ],

        sharex=axA,
    )

    fig.subplots_adjust(

        left=0.10,

        right=0.77,

        bottom=0.13,

        top=0.91,
    )

    # ========================================================
    # CURVA DE LUZ
    #
    # TRUE:
    #   lw = 2
    #   detrás
    #
    # FIT:
    #   lw = 1
    #   delante
    # ========================================================

    if logA:

        # ----------------------------------------------------
        # MODELO VERDADERO
        # ----------------------------------------------------

        axA.plot(

            t,

            np.log10(
                A
            ),

            color=COLOR_TRUE,

            linestyle="-",

            lw=LW_TRUE,

            alpha=1.0,

            label="Xallarap",

            zorder=3,
        )

        # ----------------------------------------------------
        # PSPL FIT
        # ----------------------------------------------------

        axA.plot(

            t,

            np.log10(
                A_fit
            ),

            color=COLOR_FIT,

            linestyle=LS_FIT,

            lw=LW_FIT,

            alpha=1.0,

            label="PSPL fit",

            zorder=6,
        )

        axA.set_ylabel(
            r"$\log_{10}A$"
        )

    else:

        # ----------------------------------------------------
        # MODELO VERDADERO
        # ----------------------------------------------------

        axA.plot(

            t,

            A,

            color=COLOR_TRUE,

            linestyle="-",

            lw=LW_TRUE,

            alpha=1.0,

            label="Xallarap",

            zorder=3,
        )

        # ----------------------------------------------------
        # PSPL FIT
        # ----------------------------------------------------

        axA.plot(

            t,

            A_fit,

            color=COLOR_FIT,

            linestyle=LS_FIT,

            lw=LW_FIT,

            alpha=1.0,

            label="PSPL fit",

            zorder=6,
        )

        axA.set_ylabel(
            r"$A$"
        )

    # ========================================================
    # t0
    # ========================================================

    axA.axvline(

        t0_true,

        linestyle=":",

        color="0.3",

        lw=0.8,

        alpha=0.65,

        zorder=1,
    )

    axR.axvline(

        t0_true,

        linestyle=":",

        color="0.3",

        lw=0.8,

        alpha=0.65,

        zorder=1,
    )

    # ========================================================
    # Mínimos geométricos
    # ========================================================

    axA.axvline(

        delay_info[
            "t_peak_geom_1"
        ],

        color=COLOR_TRUE,

        linestyle=LS_SOURCE1,

        lw=0.7,

        alpha=0.20,

        zorder=1,
    )

    axA.axvline(

        delay_info[
            "t_peak_geom_2"
        ],

        color=COLOR_TRUE,

        linestyle=LS_SOURCE2,

        lw=0.7,

        alpha=0.20,

        zorder=1,
    )

    # ========================================================
    # Leyenda
    # ========================================================

    axA.legend(

        loc="upper left",

        ncol=2,

        frameon=False,

        fontsize=10,

        handlelength=2.5,

        columnspacing=1.2,

        borderaxespad=0.5,
    )

    # ========================================================
    # Grilla
    # ========================================================

    axA.grid(

        which="major",

        alpha=0.10,

        linewidth=0.5,
    )

    axR.grid(

        which="major",

        alpha=0.10,

        linewidth=0.5,
    )

    # ========================================================
    # MÉTRICAS
    # ========================================================

    if out[
        "t_dev_truncated"
    ]:

        tdev_text = (
            rf"$t_{{\rm dev}}\gtrsim"
            rf"{out['t_dev_days']:.3g}"
            r"\,\mathrm{d}$"
        )

    else:

        tdev_text = (
            rf"$t_{{\rm dev}}="
            rf"{out['t_dev_days']:.3g}"
            r"\,\mathrm{d}$"
        )

    metrics_text = (

        r"$\mathrm{RMS}(|\Delta A|)="
        rf"{out['rms_abs_resid_A']:.2e}$"

        "\n"

        r"$R_{\rm max}="
        rf"{out['max_abs_resid_A']:.2e}$"

        "\n"

        + tdev_text
    )

    axA.text(

        0.025,

        0.13,

        metrics_text,

        transform=axA.transAxes,

        va="bottom",

        ha="left",

        fontsize=9.5,

        linespacing=1.25,

        bbox=dict(

            boxstyle="round,pad=0.28",

            facecolor="white",

            edgecolor="0.75",

            linewidth=0.6,

            alpha=0.92,
        ),

        zorder=30,
    )

    # ========================================================
    # Parámetros
    # ========================================================

    param_text = (
        build_param_text(

            out=out,

            P_days=P_days,

            tE_true=tE_true,

            u0_true=u0_true,

            xiE=xiE,

            use_kepler=use_kepler,

            Mtot_source_Msun=Mtot_source_Msun,

            M_lens_Msun=M_lens_Msun,

            q_mass=q_mass,

            qflux=qflux,
        )
    )

    fig.text(

        0.795,

        0.82,

        param_text,

        va="top",

        ha="left",

        fontsize=10,

        linespacing=1.2,
    )

    # ========================================================
    # RESIDUALES
    # ========================================================

    axR.plot(

        t,

        resid_A,

        color=COLOR_TRUE,

        lw=1.2,

        zorder=3,
    )

    axR.axhline(

        0.0,

        linestyle="--",

        linewidth=0.7,

        color="0.3",

        alpha=0.7,

        zorder=1,
    )

    axR.set_ylabel(
        r"$\Delta A$"
    )

    axR.set_xlabel(
        r"$t$"
    )

    # ========================================================
    # Minor ticks
    # ========================================================

    axA.yaxis.set_minor_locator(
        AutoMinorLocator()
    )

    axR.yaxis.set_minor_locator(
        AutoMinorLocator()
    )

    # ========================================================
    # Ticks temporales dinámicos
    # ========================================================

    set_time_ticks(

        axR,

        t0_true=t0_true,

        tE_true=tE_true,

        window_k=window_k,

        max_labels=7,
    )

    axA.tick_params(

        axis="both",

        which="major",

        labelsize=11,
    )

    axR.tick_params(

        axis="both",

        which="major",

        labelsize=11,
    )

    axA.tick_params(
        labelbottom=False
    )

    # ========================================================
    # INSET DE TRAYECTORIAS
    # ========================================================

    axT = axA.inset_axes([

        0.53,

        0.27,

        0.54,

        0.68,
    ])

    # ========================================================
    # Einstein ring
    # ========================================================

    ph = np.linspace(

        0,

        2.0
        * np.pi,

        400,
    )

    axT.plot(

        np.cos(
            ph
        ),

        np.sin(
            ph
        ),

        color="0.25",

        linestyle=":",

        lw=0.9,

        alpha=0.8,

        label=r"$\theta_E$",

        zorder=1,
    )

    # ========================================================
    # Trayectorias extendidas
    #
    # Truth más gruesas que fit.
    # ========================================================

    axT.plot(

        s1x_long,

        s1y_long,

        color=COLOR_TRUE,

        linestyle=LS_SOURCE1,

        lw=LW_TRUE_EXT,

        alpha=0.20,

        zorder=2,
    )

    axT.plot(

        s2x_long,

        s2y_long,

        color=COLOR_TRUE,

        linestyle=LS_SOURCE2,

        lw=LW_TRUE_EXT,

        alpha=0.20,

        zorder=2,
    )

    axT.plot(

        fit_x_long,

        fit_y_long,

        color=COLOR_FIT,

        linestyle=LS_FIT,

        lw=LW_FIT_EXT,

        alpha=0.30,

        zorder=4,
    )

    # ========================================================
    # Trayectorias principales
    #
    # Source 1/2:
    #    lw = 2
    #
    # PSPL:
    #    lw = 1
    #    por delante
    # ========================================================

    axT.plot(

        s1x,

        s1y,

        color=COLOR_TRUE,

        linestyle=LS_SOURCE1,

        lw=LW_TRUE,

        alpha=1.0,

        label="Source 1",

        zorder=6,
    )

    axT.plot(

        s2x,

        s2y,

        color=COLOR_TRUE,

        linestyle=LS_SOURCE2,

        lw=LW_TRUE,

        alpha=1.0,

        label="Source 2",

        zorder=6,
    )

    axT.plot(

        fit_x,

        fit_y,

        color=COLOR_FIT,

        linestyle=LS_FIT,

        lw=LW_FIT,

        alpha=1.0,

        # label="PSPL fit",

        zorder=10,
    )

    # ========================================================
    # Flechas
    # ========================================================

    add_direction_arrows(

        axT,

        s1x,

        s1y,

        color=COLOR_TRUE,

        n_arrows=2,

        scale=0.040,

        lw=0.9,

        zorder=12,
    )

    add_direction_arrows(

        axT,

        s2x,

        s2y,

        color=COLOR_TRUE,

        n_arrows=2,

        scale=0.040,

        lw=0.9,

        zorder=12,
    )

    # Flecha PSPL por delante
    add_direction_arrows(

        axT,

        fit_x,

        fit_y,

        color=COLOR_FIT,

        n_arrows=1,

        scale=0.040,

        lw=0.7,

        zorder=14,
    )

    # ========================================================
    # Lens
    # ========================================================

    axT.scatter(

        [0],

        [0],

        marker="+",

        s=55,

        linewidth=1.2,

        color="k",

        zorder=20,

        label="Lens",
    )

    axT.axhline(

        0,

        color="0.4",

        lw=0.45,

        alpha=0.20,

        zorder=0,
    )

    axT.axvline(

        0,

        color="0.4",

        lw=0.45,

        alpha=0.20,

        zorder=0,
    )

    # ========================================================
    # ZOOM ABSOLUTO
    #
    # traj_lim = 0.25
    # -> -0.25 <= ux,uy <= +0.25
    # ========================================================

    axT.set_xlim(

        -traj_lim,

        traj_lim,
    )

    axT.set_ylim(

        -traj_lim,

        traj_lim,
    )

    # ========================================================
    # Formato inset
    # ========================================================

    axT.set_aspect(

        "equal",

        adjustable="box",
    )

    axT.set_xlabel(

        r"$u_x$",

        fontsize=12,

        labelpad=1.5,
    )

    axT.set_ylabel(

        r"$u_y$",

        fontsize=12,

        labelpad=1.5,
    )

    axT.set_title(
        ""
    )

    axT.tick_params(

        axis="both",

        which="both",

        direction="in",

        top=True,

        right=True,

        labelsize=9,

        pad=2,
    )

    axT.xaxis.set_minor_locator(
        AutoMinorLocator()
    )

    axT.yaxis.set_minor_locator(
        AutoMinorLocator()
    )

    axT.grid(
        False
    )

    axT.legend(

        frameon=False,

        fontsize=8.5,

        loc="upper left",

        handlelength=2.2,

        borderaxespad=0.35,

        labelspacing=0.3,
    )

    # ========================================================
    # Spines
    # ========================================================

    for ax in [
        axA,
        axR,
        axT,
    ]:

        for spine in (
            ax.spines.values()
        ):

            spine.set_linewidth(
                0.8
            )

    # ========================================================
    # Ventana temporal
    # ========================================================

    axA.set_xlim(

        t0_true
        - window_k
        * tE_true,

        t0_true
        + window_k
        * tE_true,
    )

    plt.show()


# ============================================================
# SLIDERS
# ============================================================

style = {
    "description_width": "130px"
}

layout = widgets.Layout(
    width="380px"
)


# ============================================================
# OPCIONES ANGULARES
# ============================================================

pi_options_0_2pi = [

    ("0", 0.0),

    ("π/4", np.pi / 4),

    ("π/2", np.pi / 2),

    ("3π/4", 3 * np.pi / 4),

    ("π", np.pi),

    ("5π/4", 5 * np.pi / 4),

    ("3π/2", 3 * np.pi / 2),

    ("7π/4", 7 * np.pi / 4),

    ("2π", 2 * np.pi),
]


# ============================================================
# DICCIONARIO DE SLIDERS
# ============================================================

sliders = dict(

    # ========================================================
    # MICROLENSING
    # ========================================================

    P_days=widgets.FloatLogSlider(

        value=800.0,

        base=10,

        min=np.log10(
            10
        ),

        max=np.log10(
            3000
        ),

        step=0.02,

        description="P [days]",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    t0_true=widgets.FloatSlider(

        value=0.0,

        min=-100.0,

        max=100.0,

        step=1.0,

        description="t0 [days]",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    u0_true=widgets.FloatLogSlider(

        value=0.01,

        base=10,

        min=-3,

        max=np.log10(
            2
        ),

        step=0.02,

        description="u0",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    tE_true=widgets.FloatSlider(

        value=20.0,

        min=1.0,

        max=800.0,

        step=1.0,

        description="tE [days]",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    xiE=widgets.FloatSlider(

        value=0.3,

        min=0.0,

        max=3.0,

        step=0.01,

        description="xiE manual",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    # ========================================================
    # PHYSICAL
    # ========================================================

    use_kepler=widgets.Checkbox(

        value=True,

        description="Kepler mode",

        indent=False,

        layout=widgets.Layout(
            width="200px"
        ),
    ),

    Mtot_source_Msun=widgets.FloatLogSlider(

        value=1.8,

        base=10,

        min=np.log10(
            0.05
        ),

        max=np.log10(
            20.0
        ),

        step=0.02,

        description="Mtot src [Msun]",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    M_lens_Msun=widgets.FloatLogSlider(

        value=0.3,

        base=10,

        min=-12,

        max=np.log10(
            10.0
        ),

        step=0.02,

        description="M lens [Msun]",

        style=style,

        layout=layout,

        continuous_update=False,

        readout=True,

        readout_format=".1e",
    ),

    xi_source=widgets.Dropdown(

        options=[

            (
                "source 1",
                "source1",
            ),

            (
                "source 2",
                "source2",
            ),

            (
                "relative",
                "relative",
            ),
        ],

        value="source1",

        description="xi source",

        style=style,

        layout=layout,
    ),

    # ========================================================
    # GEOMETRY
    # ========================================================

    theta=widgets.SelectionSlider(

        options=pi_options_0_2pi,

        value=0.0,

        description="theta",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    phi0=widgets.SelectionSlider(

        options=pi_options_0_2pi,

        value=np.pi,

        description="phi0",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    lambda_xi=widgets.SelectionSlider(

        options=pi_options_0_2pi,

        value=0.0,

        description="lambda",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    q_mass=widgets.FloatLogSlider(

        value=0.1,

        base=10,

        min=-6,

        max=np.log10(
            2.0
        ),

        step=0.02,

        description="q mass",

        style=style,

        layout=layout,

        continuous_update=False,

        readout=True,

        readout_format=".1e",
    ),

    qflux=widgets.FloatSlider(

        value=0.45,

        min=0.0,

        max=2.0,

        step=0.01,

        description="q flux",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    # ========================================================
    # FIGURE
    # ========================================================

    window_k=widgets.FloatSlider(

        value=1.0,

        min=0.25,

        max=10.0,

        step=0.25,

        description="window [tE]",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    n_points=widgets.IntSlider(

        value=5000,

        min=1000,

        max=30000,

        step=1000,

        description="N points",

        style=style,

        layout=layout,

        continuous_update=False,
    ),

    traj_lim=widgets.FloatLogSlider(

        value=1.0,

        base=10,

        min=np.log10(
            0.03
        ),

        max=np.log10(
            10.0
        ),

        step=0.02,

        description="trajectory limit",

        style=style,

        layout=layout,

        continuous_update=False,

        readout=True,

        readout_format=".3f",
    ),

    logA=widgets.Checkbox(

        value=False,

        description="plot log10(A)",

        indent=False,

        layout=widgets.Layout(
            width="200px"
        ),
    ),
)


# ============================================================
# LAYOUT
# ============================================================

ui_microlensing = widgets.VBox([

    widgets.HTML(
        "<b>Microlensing</b>"
    ),

    sliders[
        "P_days"
    ],

    sliders[
        "t0_true"
    ],

    sliders[
        "u0_true"
    ],

    sliders[
        "tE_true"
    ],

    sliders[
        "xiE"
    ],
])


ui_kepler = widgets.VBox([

    widgets.HTML(
        "<b>Physical parameters</b>"
    ),

    sliders[
        "use_kepler"
    ],

    sliders[
        "Mtot_source_Msun"
    ],

    sliders[
        "M_lens_Msun"
    ],

    sliders[
        "xi_source"
    ],
])


ui_geometry = widgets.VBox([

    widgets.HTML(
        "<b>Binary-source geometry</b>"
    ),

    sliders[
        "theta"
    ],

    sliders[
        "phi0"
    ],

    sliders[
        "lambda_xi"
    ],

    sliders[
        "q_mass"
    ],

    sliders[
        "qflux"
    ],
])


ui_plot = widgets.VBox([

    widgets.HTML(
        "<b>Figure</b>"
    ),

    sliders[
        "window_k"
    ],

    sliders[
        "n_points"
    ],

    sliders[
        "traj_lim"
    ],

    sliders[
        "logA"
    ],
])


# ============================================================
# TABS
# ============================================================

tabs = widgets.Tab(

    children=[

        ui_microlensing,

        ui_kepler,

        ui_geometry,

        ui_plot,
    ]
)


tabs.set_title(
    0,
    "Microlensing"
)

tabs.set_title(
    1,
    "Physical"
)

tabs.set_title(
    2,
    "Geometry"
)

tabs.set_title(
    3,
    "Figure"
)


# ============================================================
# OUTPUT INTERACTIVO
# ============================================================

out_plot = widgets.interactive_output(

    plot_binary_source_widget,

    sliders,
)


display(

    tabs,

    out_plot,
)

Output()